# SALM Fine-tuning 10 · Which instrument played that note? (NSynth)

> **Task** 4-s single note → one of 11 instrument families · **Data** NSynth · **Metric** accuracy ↑
> **Result** 47.97% → **57.42%** (+19.7%) · **Time** ~1.2 h

Part of the [SALM fine-tuning series](README.md); see [00_Overview_and_Setup](00_Overview_and_Setup.ipynb) first.

## The problem: one note, no melody, no lyrics

NSynth has 300,000 isolated musical notes: one pitch, four seconds, from ~1,000 instruments grouped into 11 families
(*bass, brass, flute, guitar, keyboard, mallet, organ, reed, string, synth lead, vocal*). There's no melody to
recognize and no song to remember. The answer is purely in the **timbre**, the thing a speech encoder was never asked
to care about. The base model manages ~48% (chance ≈ 9%). It recognizes keyboard, bass and organ notes most of the
time, but almost never names a flute, and gets reed and vocal notes right only about one time in five. The goal is to
teach the encoder to hear timbre from a few thousand notes.

In [ ]:
import json, os, shlex, subprocess, sys, time
from pathlib import Path


def find_nemo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "examples" / "speechlm2" / "finetuning").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the NeMo source tree from {start}. Open this notebook from inside the repo checkout.")


NEMO_ROOT = find_nemo_root(Path.cwd())
FT = NEMO_ROOT / "examples" / "speechlm2" / "finetuning"              # the tools used below
WORK = Path(os.environ.get("SALM_FT_WORK", NEMO_ROOT / "tutorials" / "speechlm2" / "finetuning" / "workdir")).absolute()
os.environ["SALM_FT_WORK"] = str(WORK)                                # data, manifests, experiments, exports
os.environ.pop("MPLBACKEND", None)  # the kernel's inline plotting backend would break matplotlib in child processes
if not os.environ.get("SALM_CHECKPOINT"):
    raise RuntimeError(
        "SALM_CHECKPOINT is not set. Point it at your SALM checkpoint: a directory in HuggingFace format "
        "exported by SALMAutomodel. Run `export SALM_CHECKPOINT=/path/to/checkpoint` before "
        "starting Jupyter, or set os.environ['SALM_CHECKPOINT'] in a cell above this one."
    )
CKPT = Path(os.environ["SALM_CHECKPOINT"]).expanduser().absolute()  # base SALM checkpoint (HuggingFace format)
if not CKPT.is_dir():
    raise RuntimeError(f"SALM_CHECKPOINT={CKPT} is not a directory. It must be a SALM checkpoint in HuggingFace format.")
PYTHON = os.environ.get("TRAIN_PYTHON", "python")                     # NeMo training environment
VLLM_PYTHON = os.environ.get("VLLM_PYTHON", "python")                 # vLLM 0.28 environment (ideally a separate one)
M, EXP, EXPORT, EVAL = (WORK / d for d in ("manifests", "experiments", "exports", "eval"))
q = shlex.quote                                                        # shell-quote prompts (keeps non-ASCII intact)
for d in (M, EXP, EXPORT, EVAL, WORK / "logs", WORK / "configs"):
    d.mkdir(parents=True, exist_ok=True)


RETRIES = int(os.environ.get("SALM_FT_RETRIES", 3))                   # raise when other jobs share the GPU


def sh(cmd, log=None, retries=1):
    """Run a shell command from the NeMo root. With `log`, output goes to a file and only its tail is shown."""
    for attempt in range(1, retries + 1):
        print(f"$ {cmd}" + (f"   > {log}" if log else ""), flush=True)
        if log:
            with open(WORK / "logs" / log, "w") as f:
                rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
            tail = (WORK / "logs" / log).read_text(errors="ignore").splitlines()
            print("\n".join(l for l in tail[-400:] if l.startswith(("[eval]", "[export]", "[scores]", "[merge]", "wrote")))[-3000:])
        else:
            rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT).returncode
        if rc == 0:
            return
        if attempt < retries:  # e.g. a vLLM engine starting while another job allocates GPU memory
            print(f"(attempt {attempt} failed, retrying in 90 s)")
            time.sleep(90)
    raise RuntimeError(f"command failed (rc={rc}): {cmd}" + (f"; see {WORK / 'logs' / log}" if log else ""))


def evaluate(model, manifest, name, task_args, extra=""):
    """Decode `manifest` with vLLM (greedy) and return the metric summary.

    Re-running a cell reuses a previous result only if its fingerprint (model, manifest, prompt, decoder and scoring
    settings, code revision) matches; a result produced from different inputs is an error, never silently reused.
    """
    out = EVAL / f"{name}.jsonl"
    sh(f"{VLLM_PYTHON} {FT}/vllm_task_eval.py --model {model} --manifest {manifest} --out {out} {task_args} {extra} "
       f"--reuse", log=f"{name}.log", retries=RETRIES)
    return json.load(open(f"{out}.summary.json"))


def tidy(keep, name):
    """Free disk: delete this tutorial's other merged exports (~63 GB each) and leftover training checkpoints."""
    import shutil

    for p in EXPORT.glob(f"{name}_*"):
        if p != keep:
            shutil.rmtree(p)
    shutil.rmtree(EXP / name / "checkpoints", ignore_errors=True)
    print(f"kept {keep}; removed the other {name} exports and checkpoints")


def samples(name, n=3, keys=("text", "pred_text")):
    """Print a few decoded examples."""
    rows = [json.loads(l) for l in open(EVAL / f"{name}.jsonl")]
    for r in rows[:n]:
        print("  |  ".join(f"{k}: {str(r.get(k))[:110]}" for k in keys))


def result(name, value, expected, tol):
    """Compare a number with the expected result, and record it in results.json."""
    ok = abs(value - expected) <= tol
    print(f"{'✅' if ok else '❌'} {name}: {value:.4f}  (expected: {expected} ± {tol})")
    res = WORK / "results.json"
    d = json.loads(res.read_text()) if res.exists() else {}
    d[name] = {"value": value, "expected": expected, "tolerance": tol, "ok": ok}
    res.write_text(json.dumps(d, indent=1))
    return ok


print("NeMo root:", NEMO_ROOT, "\nwork dir: ", WORK, "\ncheckpoint:", CKPT)

## 1. Data: sample, don't take the first shards

The full training split is 289k notes in 75 shards of ~375 MB, and this recipe uses ~6k. Taking the first two shards
can return only two instrument families if the shards are sorted (see tutorial 06 for how that ends). The prep script
reads **random row groups from 30 random shards** instead. The mix is naturally imbalanced (bass ≫ synth lead) but
covers all 11 families. The official test set has 10: no synth lead.

In [ ]:
d = M / "nsynth"
sh(f"{PYTHON} {FT}/prepare_benchmarks.py nsynth", log="prep_nsynth.log")
sh(f"{PYTHON} {FT}/prepare_benchmarks.py librispeech", log="prep_ls.log")
import collections
print(collections.Counter(json.loads(l)["text"] for l in open(d / "train.json")).most_common())

## 2. Baseline

In [ ]:
TASK = f"--task cls --max-tokens 16 --labels {d}/labels.txt"
P2 = "{context} Reply with the label only."
base = {}
for p, extra in (("p1", ""), ("p2", f"--prompt-template {q(P2)}")):
    for s in ("dev", "test"):
        base[p, s] = 100 * evaluate(CKPT, d / f"{s}.json", f"inst_base_{p}_{s}", TASK, extra)["score"]
    print(f"{p}  dev {base[p, 'dev']:5.2f}%   test {base[p, 'test']:5.2f}%")
BEST_P = max(("p1", "p2"), key=lambda p: base[p, "dev"])  # prompt chosen on dev
BASELINE = base[BEST_P, "test"]  # its test score: the baseline fine-tuning is compared with
ORACLE = max(base[p, "test"] for p in ("p1", "p2"))  # best prompt chosen on test: an upper bound
print(f"baseline prompt (chosen on dev): {BEST_P}; test-oracle (best prompt on test): {ORACLE:.2f}")
print(f"\nbaseline = {BASELINE:.2f}%")
pairs = " ".join(f"{d}/{s}.json {d}/{s}_ft.json" for s in ("train", "dev", "test"))
sh(f"{PYTHON} {FT}/set_context.py --template '{{context}}' {pairs}")   # train with p1 (within 0.3 points of p2 on dev)

## 3. Config

In [ ]:
sh(f"{PYTHON} {FT}/epoch_batches.py {d}/train_ft.json 12000 --tokenizer {CKPT}")
CFG = WORK / "configs" / "inst.yaml"
sh(f"{PYTHON} {FT}/make_ft_config.py --checkpoint {CKPT} --prompt manifest --out {CFG} --exp-dir {EXP / 'inst'} "
   f"--train-manifest {d}/train_ft.json --val-manifest {d}/dev_ft.json --val-name inst_dev "
   f"--train-encoder --train-proj --lr 1e-4 --warmup-steps 40 --max-steps 400 "
   f"--batch-tokens 12000 --limit-train-batches 40 --val-every-n-epochs 2 --limit-val-batches 50 --save-top-k -1")

## 4. Train (~25 min)

In [ ]:
sh(f"{PYTHON} {FT}/run_training.py --config {CFG} --preflight --smoke --keep-steps 240 400")

## 5. Select on dev: an early winner, for once

Expected dev accuracy: **step 240 → 57.4%**, step 400 → 55.9%. Unlike most tutorials in this series, the later
checkpoint is *worse*, and `val_loss` also bottoms out at step 240: with ~6k notes the model starts memorizing
individual instruments. This is why you decode several checkpoints instead of assuming "last is best". Expect the
test accuracy to vary by about ±2-3 points from run to run.

In [ ]:
dev = {}
for step in (240, 400):
    out = EXPORT / f"inst_s{step}"
    sh(f"{PYTHON} {FT}/export_checkpoint.py --exp-dir {EXP / 'inst'} --step {step} --base-checkpoint {CKPT} "
       f"--out {out} --delete-ckpt", log=f"export_inst_{step}.log")
    dev[step] = 100 * evaluate(out, d / "dev_ft.json", f"inst_s{step}_dev", TASK)["score"]
    print(f"step {step}: dev accuracy {dev[step]:.2f}%")
BEST = max(dev, key=dev.get)

## 6. Test, once

In [ ]:
MODEL = EXPORT / f"inst_s{BEST}"
ft = evaluate(MODEL, d / "test_ft.json", "inst_TEST", TASK)
guard = evaluate(MODEL, M / "librispeech" / "test_clean.json", "inst_librispeech", "--task asr --normalizer english")
FT_ACC = 100 * ft["score"]
print(f"test accuracy: base {BASELINE:.2f}% → fine-tuned {FT_ACC:.2f}% ({(FT_ACC - BASELINE) / BASELINE:+.1%}); LibriSpeech {100 * guard['score']:.2f}%")
print("recall per family:", {k: round(v, 2) for k, v in sorted(ft["per_class_recall"].items(), key=lambda kv: -kv[1])})
result("10 baseline test accuracy", BASELINE, 47.97, 0.5)
result("10 fine-tuned test accuracy", FT_ACC, 57.42, 3.0)

Expected results:

| | test accuracy | LibriSpeech WER |
|---|---:|---:|
| base, prompt chosen on dev (p2) | 47.97% | 1.54% |
| **fine-tuned** | **57.42%** | 1.73% |

**+19.7% relative.** The two prompts are within 0.3 points on dev. The other one (p1) scores 49.90% on test, the
test-oracle; against it the gain is +15.1%. The biggest gains are on families the base model
missed: vocal (20% → 58% recall), guitar (31% → 59%) and string (33% → 53%). Flute (17%), reed (37%) and brass (40%)
stay hard. More notes per family (this recipe uses 2% of the training set) is the obvious next step.

## Tidy up

Keep the selected model; delete the other exports (~63 GB each) and any leftover training checkpoints.

In [ ]:
tidy(keep=MODEL, name="inst")

## Takeaways

1. **Sample randomly across shards** when you subsample a big dataset.
2. **"Last checkpoint is best" is a tendency, not a rule.** Here the earlier one won by 1.5 points.
3. **Timbre is learnable** by a speech encoder, from a few thousand notes.